# UK data cleaning notebook

- since there were data only quarterly for the basic economic indicators, we tried to go around with it and copy it monthly to fit into montly calendar panels


In [1]:
import pandas as pd

# file with the original ONS tables
XLSX_PATH = "quarterlynationalaccountsdatatables.xlsx"


def is_quarter_label(x) -> bool:
    # checks if the value looks like 2013Q1, 2014Q2, etc.
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return False
    s = str(x).strip()
    if not s:
        return False
    s = s.replace(" ", "")
    # Accept "2013Q1" or "2013Q4" (and also "2013 Q1" -> becomes "2013Q1")
    if len(s) != 6:
        return False
    year = s[:4]
    q = s[4:]
    if not year.isdigit():
        return False
    if q not in ("Q1", "Q2", "Q3", "Q4"):
        return False
    return True

def to_period_q(x) -> pd.Period:
    # changes the quarter text into pandas quarterly period format
    s = str(x).strip().replace(" ", "")
    year = int(s[:4])
    q = int(s[-1])  # Q1..Q4
    return pd.Period(year=year, quarter=q, freq="Q")

def find_row_index(df, text, col=0):
    # finds the row where a specific table label appears
    for i in range(len(df)):
        v = df.iat[i, col]
        if isinstance(v, str) and v.strip() == text:
            return i
    return None

def extract_table2_quarterly(sheet_name: str) -> pd.DataFrame:
    # main function for taking quarterly data from one sheet
    """
    Extracts 'Table 2: Quarterly' from these ONS-style sheets.
    Returns a dataframe with:
      period (Period[Q]) + columns named by the row 'Time period and dataset code row'
    """
    # reads the Excel sheet without fixed headers because the layout is custom
    df = pd.read_excel(XLSX_PATH, sheet_name=sheet_name, header=None, engine="openpyxl")

    # finds where the quarterly table starts
    t2 = find_row_index(df, "Table 2: Quarterly", col=0)
    if t2 is None:
        raise ValueError(f"Could not find 'Table 2: Quarterly' in {sheet_name}")

    titles_row = t2 + 1 
    dataset_row = t2 + 2

    ds = None
    for i in range(titles_row + 1, min(titles_row + 6, len(df))):
        v = df.iat[i, 0]
        if isinstance(v, str) and v.strip() == "Dataset identifier":
            ds = i
            break
    if ds is None:
        ds = dataset_row  # fallback

    # takes the column names from the title row
    titles = [str(x).strip() if not (isinstance(x, float) and pd.isna(x)) else "" for x in df.iloc[titles_row].tolist()]

    data_start = ds + 1
    # collect only the rows that really contain quarter values
    rows = []
    for i in range(data_start, len(df)):
        v0 = df.iat[i, 0]
        if not is_quarter_label(v0):
            # stop when the quarter column ends
            # (once quarterly table ends, col0 becomes NaN/blank/other text)
            if rows:
                break
            else:
                continue

        row = df.iloc[i].tolist()
        rows.append(row)

    # build a clean dataframe from the extracted rows
    out = pd.DataFrame(rows, columns=titles)
    out = out.rename(columns={out.columns[0]: "quarter"})
    out["period"] = out["quarter"].apply(to_period_q)
    out = out.drop(columns=["quarter"])

    # turn numeric columns into numbers where possible
    for c in out.columns:
        if c != "period":
            out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=["period"])
    return out

# helper for choosing a column by text

def pick_col(df, contains_all):
    """
    Pick first column whose name contains all substrings (case-insensitive).
    """
    contains_all_lower = [s.lower() for s in contains_all]
    for c in df.columns:
        if c == "period":
            continue
        name = str(c).lower()
        ok = True
        for s in contains_all_lower:
            if s not in name:
                ok = False
                break
        if ok:
            return c
    return None

#picking columns
# real GDP + GDP implied deflator
a1 = extract_table2_quarterly("A1 AGGREGATES")
col_real_gdp = pick_col(a1, ["gross domestic product", "chained volume"])
col_gdp_defl = pick_col(a1, ["gross domestic product", "implied deflator"])

# total compensation of employees (CoE)
d = extract_table2_quarterly("D INCOME")
col_total_coe = pick_col(d, ["total coe"]) or pick_col(d, ["compensation", "employees"])

#construction + business services & finance + total services
b1 = extract_table2_quarterly("B1 CVM OUTPUT")
col_construction = pick_col(b1, ["construction"])
col_bus_fin = pick_col(b1, ["business services", "finance"])
col_total_services = pick_col(b1, ["total service"])

# GDP per head (chained volume)
p = extract_table2_quarterly("P GDP per head")
col_gdp_ph = pick_col(p, ["gdp per head", "chained volume"]) or pick_col(p, ["gross domestic product per head", "chained volume"])

#fombining and filtering from 2013q1

out = pd.DataFrame({"period": a1["period"]})

if col_real_gdp: out["real_gdp_cvm"] = a1[col_real_gdp]
if col_gdp_defl: out["gdp_implied_deflator"] = a1[col_gdp_defl]

if col_total_coe:
    out = out.merge(d[["period", col_total_coe]].rename(columns={col_total_coe: "total_compensation_of_employees"}),
                    on="period", how="left")

if col_construction:
    out = out.merge(b1[["period", col_construction]].rename(columns={col_construction: "construction_real_output"}),
                    on="period", how="left")

if col_bus_fin:
    out = out.merge(b1[["period", col_bus_fin]].rename(columns={col_bus_fin: "business_services_and_finance_real_output"}),
                    on="period", how="left")

if col_total_services:
    out = out.merge(b1[["period", col_total_services]].rename(columns={col_total_services: "total_services_real_output"}),
                    on="period", how="left")

if col_gdp_ph:
    out = out.merge(p[["period", col_gdp_ph]].rename(columns={col_gdp_ph: "gdp_per_head_real"}),
                    on="period", how="left")

# Keep from 2013Q1 onwards
out = out[out["period"] >= pd.Period("2013Q1", freq="Q")].sort_values("period").reset_index(drop=True)

# Save simple output
out.to_csv("uk_key_stability_indicators_quarterly_from_2013.csv", index=False)

print("Saved: uk_key_stability_indicators_quarterly_from_2013.csv")
print(out.head(8))



/Users/petrahalaszova/miniconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Saved: uk_key_stability_indicators_quarterly_from_2013.csv
   period  real_gdp_cvm  gdp_implied_deflator  \
0  2013Q1          84.3                  75.9   
1  2013Q2          84.8                  76.0   
2  2013Q3          85.5                  76.6   
3  2013Q4          86.0                  76.8   
4  2014Q1          86.7                  77.2   
5  2014Q2          87.5                  77.1   
6  2014Q3          88.3                  78.1   
7  2014Q4          88.8                  77.8   

   total_compensation_of_employees  construction_real_output  \
0                           212454                      83.7   
1                           219443                      85.4   
2                           220069                      87.3   
3                           221990                      87.3   
4                           224324                      87.5   
5                           223540                      88.3   
6                           223020                 

In [2]:
#from quarterly to monthly

import pandas as pd

# input and output file names
IN_CSV = "uk_key_stability_indicators_quarterly_from_2013.csv"
OUT_CSV = "uk_key_stability_indicators_monthly_from_2013.csv"

dfq = pd.read_csv(IN_CSV)

def quarter_to_month_starts(qstr: str):
    # changes one quarter into its three starting months
    qstr = str(qstr).strip()
    year = int(qstr[:4])
    q = int(qstr[-1])  # 1..4
    start_month = (q - 1) * 3 + 1
    # returns 3 monthly timestamps for the quarter
    return pd.date_range(f"{year}-{start_month:02d}-01", periods=3, freq="MS")

# repeat each quarterly value for all 3 months in that quarter
rows = []
value_cols = [c for c in dfq.columns if c != "period"]

# go through each quarterly row and copy it into 3 monthly rows
for _, r in dfq.iterrows():
    months = quarter_to_month_starts(r["period"])
    for m in months:
        out_row = {"month": m.strftime("%Y-%m")}
        for c in value_cols:
            out_row[c] = r[c]
        rows.append(out_row)

dfm = pd.DataFrame(rows)

# sort the final monthly data just to keep the order clean
dfm["month_dt"] = pd.to_datetime(dfm["month"] + "-01")
dfm = dfm.sort_values("month_dt").drop(columns=["month_dt"]).reset_index(drop=True)

# save the final monthly dataset
dfm.to_csv(OUT_CSV, index=False)

print(f"Saved: {OUT_CSV}")
print(dfm.head(12))


Saved: uk_key_stability_indicators_monthly_from_2013.csv
      month  real_gdp_cvm  gdp_implied_deflator  \
0   2013-01          84.3                  75.9   
1   2013-02          84.3                  75.9   
2   2013-03          84.3                  75.9   
3   2013-04          84.8                  76.0   
4   2013-05          84.8                  76.0   
5   2013-06          84.8                  76.0   
6   2013-07          85.5                  76.6   
7   2013-08          85.5                  76.6   
8   2013-09          85.5                  76.6   
9   2013-10          86.0                  76.8   
10  2013-11          86.0                  76.8   
11  2013-12          86.0                  76.8   

    total_compensation_of_employees  construction_real_output  \
0                            212454                      83.7   
1                            212454                      83.7   
2                            212454                      83.7   
3                  